In [1]:
import json
from datetime import datetime, timedelta
from langchain_google_genai import ChatGoogleGenerativeAI

from crewai import Agent, Task, Crew, Process
from crewai_tools import CSVSearchTool


from dotenv import load_dotenv
import os

/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/crewai/telemetry/telemetry.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/alysson/Desktop/Rocketseat/ia-para-devs/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:949: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


In [2]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.7,
    api_key=api_key,
    max_retries=5
)

In [3]:
csvWalletTool = CSVSearchTool(
    csv="./Wallet/wallet.csv",
    description="A tool to search the customer wallet CSV. You MUST provide the 'search_query' argument containing the stock ticker.",
    config={
        "llm": {
            "provider": "google",
            "config": {
                "model": "gemini-3.6-flash",
            },
        },
        "embedder": {
            "provider": "google",
            "config": {
                "model": "models/gemini-embedding-001",
            },
        },
    },
)

In [4]:
customerManager = Agent(
    role="Customer Stocks Manager",
    goal="Get the customer question about the stock {ticket} and search the customer wallet CSV file for the stocks.",
    backstory=""" You're the managet of the customer investiments wallet.
     You're the client first contact and you provide the other analystis with the necessary stock ticket and wallet information. 
    """,
    llm=llm,
    verbose=True,
    max_iter=5,
    tools=[csvWalletTool],
    allow_delegation=False,
    memory=True
)

In [5]:
getCustomerWallet = Task(
    description="""Search for the {ticket} in the CSV file using the provided tool. 
     IMPORTANT: When calling the CSV tool, you MUST pass the argument 'search_query' with the value of the {ticket}.
     Determine if the stock is in the customer's wallet. If it is, provide the mean price paid and the total number of shares owned.
    """,
    expected_output="If the customer owns the stock, provide the mean price paid and the total number of shares.",
    agent=customerManager
)

In [6]:
stockPriceAnalyst = Agent(
    role="Senior Stock Price Analyst",
    goal="Find the {ticket} stock price and analyses its trends. Compare with the price that the customer paid.",
    backstory="""" You're a highly experienced in analyzing the price of specific stock and 
     make predictions about its future price.
    """,
    llm=llm,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [7]:
import yfinance as yf

def fetch_stock_price(ticket):
    end_date = datetime.today()
    start_date = end_date - timedelta(days=365)
    stock = yf.download(ticket, 
                        start=start_date.strftime("%Y-%m-%d"), 
                        end=end_date.strftime("%Y-%m-%d"))
    return stock

In [8]:
from langchain.tools import Tool

yahoo_finance_tool = Tool(
    name="Yahoo Finance Tool",
    description="Fetches stocks prices for {ticket} from the last year about a specific company from the Yahoo finance API.",
    func=lambda ticket: fetch_stock_price(ticket)
)

In [9]:
getStockPrice = Task(
    description="Analyze the stock {ticket} price history and create a trend analyses of up, down or sideways.",
    expected_output="""Specify the current trend stocks price - Up, down or sideway
     e.g. stock='AAPL, price UP'.
    """,
    tools= [yahoo_finance_tool],
    agent=stockPriceAnalyst
)

In [10]:
newsAnalyst = Agent(
    role="News Analyst",
    goal="""Create a short summary of the market related to the stock {ticket} company.
     Provide a market Fear & Greed Index score about the company.
     For each requested stock asset, specify a number between 0 and 100, where 0 is extreme fear and 100 ins extreme greed.
    """,
    backstory="""You are highly experienced in analyzing market trends and news for more then 10 years.
     You're also a master level analyst in the human psychology.
     You understand the news, ther title and information, but you look at those with a health dose of skeptcism.
     You consider the source of the news articles.
    """,
    llm=llm,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [11]:
from langchain_community.tools import DuckDuckGoSearchResults

searchTool = DuckDuckGoSearchResults(backend="news", num_results=10)

In [12]:
getNews = Task(
    description="""Use the search tool to find recent news about the {ticket} stock.
    The current date is {current_date}.
    Compose the results into a helpful report.""",
    expected_output="""A summary of the overall market and a one-paragraph summary for the {ticket} asset.
    Include the Fear/Greed score based on the news. Format:
    <STOCK TICKET>
    <SUMMARY BASED ON NEWS>
    <FEAR/GREED SCORE>""",
    tools=[searchTool],
    agent=newsAnalyst
)

In [13]:
stockRecommender = Agent(
    role="Chief Stock Analyst",
    goal="""Get the data from the customer currently stocks, the provided input of stock price trends and
     the stock news to provide a recommendation: Buy, Sell or Hold the stock.
    """,
    backstory="""You're the leader of the stock analyst team. You have a great performance in the past 20 years in stock recommendation.
     with all your team information, you are able to provide the best recommendation for the customer to achieve 
     the maximum value creation.
    """,
    llm=llm,
    verbose=True,
    max_iter=5,
    allow_delegation=True,
    memory=True
)

In [14]:
recommendStock = Task(
    description="""Use the stock prince trend, the stock news report and the customers mean price of the {ticket} to
     provide a recommendation: Buy, Sell or Hold. If the previous reports are not well provided you can delegate back to
     the specific analyst to worg again in the their task.
    """,
    expected_output="""A brief paragraph paragraph with the summary of the reason for recommendation and the
     motivation it self in one of the three possible outputs: Buy, Sell or Hold. Use the format:
     <SUMMARY OF REASONS>
     <RECOMMENDATION>
    """,
    agent=stockRecommender,
    context=[getCustomerWallet, getStockPrice, getNews]
)

In [15]:
copyWritter = Agent(
    role="Stock Content Writer",
    goal="""Write an insightfull compelling and informative 3 paragraph long newsletter
     based on the stock price report, the news report and the recommendation report.
    """,
    backstory="""You're a unbeliveble copy writter that understand complex financel concepts
    and explain for a dummie audience.
    You create complelling stories and narratives that resonate with the audience.
    """,
    llm=llm,
    verbose=True,
    max_iter=5,
    allow_delegation=False,
    memory=True
)

In [16]:
writeNewsletter = Task(
    description="""Use the stock price trend, the stock news report and the stock recommendation to write an insightfull compelling and informative 6 paragraph long newsletter.
    Focus on the stock price treind, the news and fear/greed score and the summary reason for the recommendation.
    Include the recommendation in the newsletter.
    """,
    expected_output="""An  eloquent 6 paragraph newsletter formated as Markdown in an easy readable manner. It should contain:
     - Introduction - set the overal picture
     - Main part - provides the meat of the analysis including stock price trend, the news, the fear/greed score and the summary reason for the recommendation
     - 3 bullets of the main summary reason of the recommendation.
     - Recommendation it self.
    """,
    agent=copyWritter,
    context=[getStockPrice, getNews, recommendStock]
)

In [17]:
crew = Crew(
    agents=[customerManager, stockPriceAnalyst, newsAnalyst, stockRecommender, copyWritter],
    tasks=[getCustomerWallet, getStockPrice, getNews, recommendStock, writeNewsletter],
    verbose=2,
    process=Process.hierarchical,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15,
    max_rpm=10 # compatible with free tier
)

In [18]:
current_date = datetime.now().strftime("%Y-%m-%d")

result=crew.kickoff(inputs={"ticket": "AMZN",
                            "current_date": current_date})

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Search for the AMZN in the CSV file using the provided tool. 
     IMPORTANT: When calling the CSV tool, you MUST pass the argument 'search_query' with the value of the AMZN.
     Determine if the stock is in the customer's wallet. If it is, provide the mean price paid and the total number of shares owned.
    


> Entering new CrewAgentExecutor chain...
Thought: The current task requires searching for 'AMZN' in the CSV file, determining if the stock is in the customer's wallet, and if so, providing the mean price paid and the total number of shares owned. Among my available co-workers, the "Customer Stocks Manager" is the most appropriate person to handle customer portfolio and stock CSV checks. I need to delegate this specific task to the Customer Stocks Manager with all necessary context.

Action: Delegate work to co-worker
Action Input: {"coworker": "Customer Stocks Manager", "task": "Search for the stock ticker AMZN in 

2026-08-02 22:22:16,090 - 8518574464 - before_sleep.py-before_sleep:65 - WARNING: Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite
Please retry in 43.994951119s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.5-flash-lite"
  }

Thought: I now know the final answer
Final Answer: The stock is AMZN, the mean price paid by the customer is 146.29, and the number of shares held is 4.

> Finished chain.
 

The stock is AMZN, the mean price paid by the customer is 146.29, and the number of shares held is 4.



2026-08-02 22:22:19,068 - 8518574464 - before_sleep.py-before_sleep:65 - WARNING: Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite
Please retry in 40.993595964s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.5-flash-lite"
  }

Thought: I have gathered all the necessary information: the stock price trend is UP, driven by robust AWS growth and e-commerce margins; the news report highlights strong momentum tempered by macro and regulatory risks with a Greed score of 74; and the customer's mean price paid for AMZN is 146.29 with 4 shares held. I am now ready to formulate the final recommendation.

Final Answer: Amazon (AMZN) demonstrates robust upward momentum supported by accelerating AWS growth driven by enterprise AI adoption and strong margins in core e-commerce and global logistics. Although mainstream financial media hype creates a euphoric market sentiment with a Fear & Greed score of 74, underlying macroeconomic headwinds, antitrust scrutiny, and cautious consumer spending warrant a healthy dose of skepticism. Given that the customer's mean price paid is 146.29—well below current trading levels—the position sits on a solid unrealized gain. Therefore, the recommendation is to hold the position to capture 

2026-08-02 22:22:27,403 - 8518574464 - before_sleep.py-before_sleep:65 - WARNING: Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite
Please retry in 32.747612056s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.5-flash-lite"
  }

Thought: I need to have the Stock Content Writer produce the 6-paragraph newsletter based on the provided stock price trend, news, fear/greed score, summary reasons, and recommendation for AMZN.
Action: Delegate work to co-worker
Action Input: {"coworker": "Stock Content Writer", "task": "Write an insightful, compelling, and informative 6-paragraph newsletter formatted as Markdown based on the provided AMZN stock context.", "context": "The stock is AMZN and its price is trending UP. The context includes: 1) As of early August 2026, Amazon (AMZN) continues to demonstrate robust resilience in the market, with its stock price currently trending upward. Recent news highlights accelerating growth in Amazon Web Services (AWS) driven by enterprise AI adoption, alongside surprisingly strong margins in its core e-commerce and global logistics segments. While mainstream financial media is heavily hyping these quarterly beats and painting a picture of unstoppable dominance, a healthy dose of skep

In [23]:
from IPython.display import Markdown

Markdown(result["final_output"])

# Navigating the Amazon Wave: AI Momentum Meets Market Euphoria

Welcome back to your go-to financial breakdown, where we take the complex Wall Street chatter and translate it into plain English. If you’ve been keeping an eye on your portfolio lately, you’ve probably noticed that Amazon (AMZN) is riding a serious wave. The stock is trending upward, and if your average purchase price sits around that comfortable $146.29 mark, you are currently sitting on some very handsome unrealized gains. It is easy to look at those green numbers and feel like a financial genius, but as your trusted guide, my job is to look past the shiny surface and help you understand what is actually driving this rocket ship—and whether it’s about to run out of fuel.

To understand why AMZN is climbing, we have to look at the twin engines currently powering the company: Amazon Web Services (AWS) and surprisingly resilient e-commerce margins. AWS is cashing in big on the enterprise artificial intelligence boom, as businesses everywhere scramble to integrate smart tech into their operations. At the same time, Amazon's global logistics and retail arms are running leaner and more profitably than the skeptics expected. Mainstream financial media is eating this up, painting a picture of an unstoppable titan destined for perpetual growth. But remember: headlines love a good fairy tale. Beneath the corporate PR glow, we still have to contend with stubborn macroeconomic headwinds, cautious everyday consumers tightening their purse strings, and the ever-present shadow of antitrust regulators breathing down Amazon's neck. 

Adding fuel to this upward price action is the broader market's psychological state. Right now, the Fear & Greed Index is flashing a hot 72 to 74, squarely sitting in "Greed" territory. This means retail investors are feeling euphoric, chasing momentum, and pouring money into the market with little regard for gravity. While market euphoria is fantastic for short-term price appreciation, it often stretches valuations past a comfortable point. When everyone in the room is high-fiving, it is usually the exact right moment to take a deep breath, step back, and check the fundamentals rather than getting swept away by the crowd's adrenaline.

So, where does this leave everyday investors who are enjoying the view from higher altitudes? Despite the stellar AWS growth and the smiling faces on financial TV, the official recommendation on AMZN right now is a disciplined **HOLD**. That might feel counterintuitive when a stock is trending up and the Fear & Greed meter is running hot, but investing isn't about chasing every single percentage point of a hype cycle—it's about protecting your hard-earned capital. 

Here is the breakdown of why the official recommendation is to hold your horses:
* **Overextended Market Sentiment:** With the Fear & Greed index sitting near 74, short-term enthusiasm may be outpacing reality, leaving the stock vulnerable to sudden sentiment shifts or minor earnings hiccups.
* **Lingering Regulatory Risks:** Antitrust scrutiny hasn't vanished; legal and regulatory headwinds remain a persistent structural risk that could impact future business agility and valuations.
* **Solid Existing Cushion:** For investors with a favorable average cost basis, holding allows you to capture long-term cloud and AI upside without exposing yourself to the higher risks of buying fresh at current euphoric peaks.

Ultimately, Amazon remains an absolute powerhouse of modern commerce and cloud infrastructure, and its long-term story is still incredibly bright. However, managing wealth means knowing when to dance and when to just watch the music play from a comfortable seat. Enjoy your gains, stay level-headed while the market gets greedy, and maintain your position as we watch how these macroeconomic and AI narratives unfold in the months ahead.